# Sanctions Screening & Compliance Analytics


In [ ]:
import pandas as pd
a=pd.read_csv('../data/processed/sanctions_screening_alerts.csv')
a.head()


## Match score and disposition

In [ ]:
a.groupby('disposition')['name_match_score'].agg(['count','mean','median'])


## Sanctions program analysis

In [ ]:
a.groupby('sanctions_program')['flagged_amount'].agg(['count','sum']).sort_values('sum',ascending=False)


## EDA + Feature Engineering

This section adds a simple exploratory data analysis and feature-engineering workflow to the existing **Sanctions Screening & Compliance Analytics** project.

The original match-score/disposition analysis and sanctions-program analysis above are kept unchanged.


### 1. Dataset Review

In [ ]:
# Review the sanctions screening dataset already loaded above
print("Dataset shape:", a.shape)
display(a.head())
display(a.dtypes.to_frame("data_type"))


### 2. Missing Values, Duplicates & Data Quality

In [ ]:
# Basic data-quality checks
print("Duplicate rows:", a.duplicated().sum())

missing = a.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if len(missing):
    display(missing.to_frame("missing_count"))
else:
    print("No missing values found.")

print("\nColumns:")
print(list(a.columns))


### 3. Summary Statistics & Range Validation

In [ ]:
# Review important numeric fields already used in this project
numeric_fields = [
    col for col in ['name_match_score', 'flagged_amount']
    if col in a.columns
]

if numeric_fields:
    display(a[numeric_fields].describe().T)

# Simple logical checks
if 'name_match_score' in a.columns:
    print("Missing name-match scores:", a['name_match_score'].isna().sum())

if 'flagged_amount' in a.columns:
    print("Negative flagged amounts:", (a['flagged_amount'] < 0).sum())


### 4. Simple Outlier Review

In [ ]:
# Review unusually high flagged amounts using the IQR method.
# In sanctions analysis, unusual values are flagged for review rather than deleted.
outlier_summary = []

if 'flagged_amount' in a.columns and pd.api.types.is_numeric_dtype(a['flagged_amount']):
    q1 = a['flagged_amount'].quantile(0.25)
    q3 = a['flagged_amount'].quantile(0.75)
    iqr = q3 - q1

    if iqr > 0:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        count = ((a['flagged_amount'] < lower) | (a['flagged_amount'] > upper)).sum()

        outlier_summary.append({
            'field': 'flagged_amount',
            'potential_outliers': int(count)
        })

display(pd.DataFrame(outlier_summary))


### 5. Feature Engineering

These features are simple and directly related to sanctions-screening review.


In [ ]:
# Work on a copy so the original analysis stays unchanged
a_fe = a.copy()
created_features = []

# Match-score band
if 'name_match_score' in a_fe.columns:
    a_fe['match_score_band'] = pd.cut(
        a_fe['name_match_score'],
        bins=[-float('inf'), 60, 80, 90, float('inf')],
        labels=['Low', 'Moderate', 'High', 'Very High']
    )
    created_features.append('match_score_band')

# Higher-match flag based on a simple 80+ threshold
if 'name_match_score' in a_fe.columns:
    a_fe['high_match_flag'] = (a_fe['name_match_score'] >= 80).astype(int)
    created_features.append('high_match_flag')

# Flagged-amount band
if 'flagged_amount' in a_fe.columns:
    a_fe['flagged_amount_band'] = pd.cut(
        a_fe['flagged_amount'],
        bins=[-float('inf'), 1000, 10000, 50000, float('inf')],
        labels=['Under 1K', '1K-10K', '10K-50K', '50K+']
    )
    created_features.append('flagged_amount_band')

# Disposition-derived escalation flag, only if the project has a disposition field
if 'disposition' in a_fe.columns:
    disposition_text = a_fe['disposition'].astype(str).str.strip().str.lower()
    a_fe['escalation_flag'] = disposition_text.str.contains(
        'escalat|true match|confirmed',
        regex=True
    ).astype(int)
    created_features.append('escalation_flag')

print("Features created:", created_features)
display(a_fe.head())


### 6. Business Rule & KPI Validation

In [ ]:
# Reconfirm the main project metrics
if 'disposition' in a_fe.columns and 'name_match_score' in a_fe.columns:
    print("Match score by disposition:")
    display(
        a_fe.groupby('disposition')['name_match_score']
        .agg(['count', 'mean', 'median'])
    )

if 'sanctions_program' in a_fe.columns and 'flagged_amount' in a_fe.columns:
    print("Sanctions program exposure:")
    display(
        a_fe.groupby('sanctions_program')['flagged_amount']
        .agg(['count', 'sum'])
        .sort_values('sum', ascending=False)
    )

if 'high_match_flag' in a_fe.columns:
    print("High-match alerts:", int(a_fe['high_match_flag'].sum()))

if 'escalation_flag' in a_fe.columns:
    print("Escalated / confirmed alerts:", int(a_fe['escalation_flag'].sum()))


### 7. Final Validation & Optional Export

In [ ]:
print("Final dataset shape:", a_fe.shape)
print("Duplicate rows:", a_fe.duplicated().sum())
print("Total missing values:", int(a_fe.isna().sum().sum()))

# Optional export for Tableau, Streamlit, or further analysis.
# a_fe.to_csv(
#     '../data/processed/sanctions_screening_alerts_enriched.csv',
#     index=False
# )

print("EDA + Feature Engineering completed.")
